# 01 - Collect a Training Dataset

This notebook shows the standard Mouse Core data collection pattern:

1. Build one or more environments with `mouse_gym.EnvConfig` and `make_group_env`.
2. Create one `Datastore` for each environment stream.
3. Step the grouped environment, merge each input and output into a row, and append rows to the matching store.
4. Publish the stores with `push_stores_to_hub` so training notebooks can load the same data later.

The environment used here is Procedural Frozen Lake, but the Mouse Core pattern is environment-agnostic. Any environment output that can be represented as dictionaries of Python values can be written to a `Datastore`.


In [ ]:
import numpy as np
import torch

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env
from mouse_core.data import Datastore, push_stores_to_hub


DATASET_ID  = "mouse-example-dataset"  # Hugging Face dataset repo for push_stores_to_hub
NUM_ENVS    = 30                         # number of environment streams in the GroupEnv
STEPS_PER_ENV = 50_000                   # rows collected for each datastore
MAX_EPISODE_STEPS = 30                   # environment-specific episode limit
EPISODES_PER_TASK = 20                   # environment-specific task length
ORACLE_PROB_END = 1.0                    # final probability of using the expert action during collection

# Total rows collected are NUM_ENVS * STEPS_PER_ENV, split across one datastore per env.


## Build Environment

`EnvConfig` is the bridge between an environment package and Mouse Core data collection. Each config names an environment, gives it a stable stream name, and passes reset/task options plus environment-specific keyword arguments.

`make_group_env(configs)` returns a `GroupEnv`. A group environment lets you step many independent environment instances with one call. The returned `env.names` are useful because they line up with outputs, metrics, and the datastores created below.

A few config fields are worth noticing when adapting this example:

- `name` becomes the stream name you can reuse for a matching `Datastore`.
- `reset_seed` and environment seeds make generated data reproducible.
- `episodes_per_task` and `task_reset_options` define where task boundaries occur. Mouse Core objectives read those boundaries from the `done` field later.
- `kwargs` are passed through to the underlying Gymnasium environment. Mouse Core does not require these specific fields; they are specific to this example environment.


In [12]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_{i}",
        reset_seed=i,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},  # forwarded to the environment at task reset
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i,
            "slippery_success_rate": 1.0,  # environment-specific option
            "permute_obs": True,      # environment-specific option
            "permute_actions": True,  # environment-specific option
            "emit_q_star": True,      # include a field we flatten for downstream objectives
        },
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)

## Create Datastores

A `Datastore` is an append-only sequence of step dictionaries backed by Hugging Face Arrow data. It does not require a fixed schema up front; the rows you append define the columns.

The common pattern is one datastore per environment stream. Keeping streams separate preserves sequential ordering inside each environment while still letting `DataLoader` mix streams during training.


In [13]:
stores = [Datastore(name=name) for name in env.names]  # one ordered store per env stream

## Collect Transitions

Each environment step has two parts:

1. The input you sent, such as `{"action": ...}`.
2. The output returned by the environment, such as `observation`, `reward`, `done`, indices, and optional `info` values.

Mouse Core training examples expect a flat row dictionary, so this notebook merges input and output into one row before appending it. If an environment returns nested `info`, flatten only the fields your objective or model will need later.

The first `env.step(...)` also performs the initial reset for each environment instance. Storing that row keeps every stream aligned from its first observed state.


In [14]:
def choose_inputs(outputs, env, oracle_prob: float) -> list[dict]:
    """Build one input dictionary per environment output."""
    random_inputs = env.sample_random_input()
    inputs = []
    for i, out in enumerate(outputs):
        if torch.rand(1).item() >= oracle_prob:  # choose an exploratory input
            inputs.append(random_inputs[i])
        else:
            action = out["info"]["q_star"].argmax()  # read an environment-provided target field
            inputs.append({"action": np.int64(action)})
    return inputs


if not (0.0 <= ORACLE_PROB_END <= 1.0):
    raise ValueError(f"ORACLE_PROB_END must be in [0, 1], got {ORACLE_PROB_END}.")

outputs = None
env.metrics.clear()
for step in range(STEPS_PER_ENV):
    # Choose the collection policy for this step.
    oracle_prob = ORACLE_PROB_END * (step / (STEPS_PER_ENV - 1))
    if outputs is None:                          # first step triggers reset
        inputs = env.sample_random_input()
    else:
        inputs = choose_inputs(outputs, env, oracle_prob)
    outputs = env.step(inputs)
    for store, inp, out in zip(stores, inputs, outputs):
        row = {**inp, **out}
        info = row.pop("info")                   # flatten nested info fields before storing
        row["info_q_star"] = info["q_star"]
        store.append(row)
    if (step + 1) % 1000 == 0:
        task_scores = [r for env_tasks in env.metrics.task_cum_rewards for r in env_tasks]
        mean_task_score = sum(task_scores) / len(task_scores) if task_scores else float("nan")
        print(
            f"step {step + 1}/{STEPS_PER_ENV} | oracle_prob={oracle_prob:.2f}"
            f" | {len(task_scores)} tasks | mean task score {mean_task_score:.2f}/{EPISODES_PER_TASK}"
        )
        env.metrics.clear()  # reset environment metrics for the next progress window

env.close()


step 1000/50000 | oracle_prob=0.00 | 113 tasks | mean task score 2.14/20
step 2000/50000 | oracle_prob=0.00 | 128 tasks | mean task score 2.28/20
step 3000/50000 | oracle_prob=0.00 | 121 tasks | mean task score 2.45/20
step 4000/50000 | oracle_prob=0.00 | 134 tasks | mean task score 1.96/20
step 5000/50000 | oracle_prob=0.00 | 121 tasks | mean task score 2.55/20
step 6000/50000 | oracle_prob=0.00 | 125 tasks | mean task score 2.54/20
step 7000/50000 | oracle_prob=0.00 | 129 tasks | mean task score 2.36/20
step 8000/50000 | oracle_prob=0.00 | 125 tasks | mean task score 2.42/20
step 9000/50000 | oracle_prob=0.00 | 118 tasks | mean task score 2.35/20
step 10000/50000 | oracle_prob=0.00 | 127 tasks | mean task score 2.32/20
step 11000/50000 | oracle_prob=0.00 | 116 tasks | mean task score 2.37/20
step 12000/50000 | oracle_prob=0.00 | 123 tasks | mean task score 2.63/20
step 13000/50000 | oracle_prob=0.00 | 129 tasks | mean task score 2.26/20
step 14000/50000 | oracle_prob=0.00 | 131 tasks

## Push to the Hub

`push_stores_to_hub` uploads all datastores to one Hugging Face dataset repository. Each datastore is saved as a named config, so downstream code can reload the same independent streams with `load_stores_from_hub`.


In [15]:
url = push_stores_to_hub(stores, repo_id=DATASET_ID, split="train", private=False)
print(f"Pushed to {url}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to micahr234/mouse-example-dataset-random (proc_frozenlake_0/train: 50000, proc_frozenlake_1/train: 50000, proc_frozenlake_2/train: 50000, proc_frozenlake_3/train: 50000, proc_frozenlake_4/train: 50000, proc_frozenlake_5/train: 50000, proc_frozenlake_6/train: 50000, proc_frozenlake_7/train: 50000, proc_frozenlake_8/train: 50000, proc_frozenlake_9/train: 50000, proc_frozenlake_10/train: 50000, proc_frozenlake_11/train: 50000, proc_frozenlake_12/train: 50000, proc_frozenlake_13/train: 50000, proc_frozenlake_14/train: 50000, proc_frozenlake_15/train: 50000, proc_frozenlake_16/train: 50000, proc_frozenlake_17/train: 50000, proc_frozenlake_18/train: 50000, proc_frozenlake_19/train: 50000, proc_frozenlake_20/train: 50000, proc_frozenlake_21/train: 50000, proc_frozenlake_22/train: 50000, proc_frozenlake_23/train: 50000, proc_frozenlake_24/train: 50000, proc_frozenlake_25/train: 50000, proc_frozenlake_26/train: 50000, proc_frozenlake_27/train: 50000, proc_frozenlake_28/train: 50000, pro